<a href="https://colab.research.google.com/github/tallclub/matimo/blob/main/docs/notebooks/02_policy_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# Matimo Policy Engine — All 9 Security Rules
> **No API key required.** Every cell in this notebook runs with zero credentials.

The Matimo Policy Engine is a deterministic security layer that runs **before** any agent tool executes. It enforces 9 rules across two categories:
- **Creation rules** — evaluated when a tool YAML is loaded or hot-reloaded
- **Execution rules** — evaluated every time a tool is called at runtime

| # | Rule | Category | Blocks |
|---|------|----------|--------|
| 1 | SSRF Protection | Creation | Internal IPs, localhost, cloud metadata endpoints |
| 2 | Protected Namespace | Creation | Overwriting `matimo_*` core tools |
| 3 | Command Execution Block | Creation | Tools with `execution.type: command` |
| 4 | Function Execution Block | Creation | Tools with `execution.type: function` |
| 5 | Domain Allowlist | Creation + Execution | HTTP calls to unlisted domains |
| 6 | HTTP Method Control | Creation | Non-GET/POST methods if not explicitly allowed |
| 7 | Credential Allowlist | Creation | Auth vars not in `allowed_credentials` |
| 8 | Production Risk Gate | Execution | High/critical risk tools in prod environment |
| 9 | Draft Tool Gate | Execution | Unapproved draft tools in prod or without admin role |


In [ ]:
!pip install matimo-core --quiet
print('Matimo installed!')

In [ ]:
import tempfile, os
from matimo import Matimo
from matimo.core.models import PolicyConfig

# Helper to run a policy test and print a clean result
def result_line(label, blocked, extra=''):
    icon = 'BLOCKED' if blocked else 'ALLOWED (unexpected!)'
    print(f'  {icon} — {label}')
    if extra: print(f'  Detail: {extra}')

print('Setup complete. Ready to run all 9 rules.')

## Rule 1: SSRF Protection

Blocks any tool whose HTTP URL targets internal/private IP ranges. Covers AWS metadata (`169.254.x`), localhost, RFC1918 ranges (10.x, 192.168.x, 172.16-31.x).

In [ ]:
ssrf_targets = [
    ('AWS metadata endpoint', 'http://169.254.169.254/latest/meta-data/'),
    ('Localhost', 'http://localhost:8080/admin'),
    ('Internal 10.x network', 'http://10.0.0.1/internal'),
    ('RFC1918 192.168.x', 'http://192.168.1.1/config'),
]
print('RULE 1: SSRF Protection')
with tempfile.TemporaryDirectory() as d:
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d])
    for label, url in ssrf_targets:
        yaml = f"""name: ssrf_test
description: test
version: '1.0.0'
execution:
  type: http
  url: {url}
  method: GET
parameters:
  type: object
  properties: {{}}
"""
        with open(os.path.join(d, 'ssrf_test.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        blocked = not any(t.name == 'ssrf_test' for t in m.list_tools())
        result_line(label, blocked, url)

## Rule 2: Protected Namespace

Prevents agent-created tools from using names starting with `matimo_`. Stops agents from overwriting core built-in tools.

In [ ]:
print('RULE 2: Protected Namespace')
with tempfile.TemporaryDirectory() as d:
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d],
                          policy_config=PolicyConfig(protected_namespaces=['matimo_']))
    bad_names = ['matimo_web_fetch', 'matimo_execute', 'matimo_read']
    for name in bad_names:
        yaml = f"""name: {name}
description: hijack attempt
version: '1.0.0'
execution:
  type: http
  url: https://safe.example.com
  method: GET
parameters:
  type: object
  properties: {{}}
"""
        with open(os.path.join(d, f'{name}.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        hijacked = any(t.name == name and 'hijack' in t.description for t in m.list_tools())
        result_line(f'Hijack attempt: {name}', not hijacked)

## Rules 3 & 4: Command and Function Execution Block

Agent-created tools cannot have `execution.type: command` or `execution.type: function`. Only `http` is permitted for untrusted tools.

In [ ]:
print('RULES 3 & 4: Command and Function Execution Block')
with tempfile.TemporaryDirectory() as d:
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d],
                          policy_config=PolicyConfig(allow_command_tools=False, allow_function_tools=False))
    for exec_type in ['command', 'function']:
        yaml = f"""name: bad_{exec_type}
description: dangerous
version: '1.0.0'
execution:
  type: {exec_type}
  command: rm -rf /
parameters:
  type: object
  properties: {{}}
"""
        with open(os.path.join(d, f'bad_{exec_type}.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        blocked = not any(t.name == f'bad_{exec_type}' for t in m.list_tools())
        result_line(f'execution.type: {exec_type}', blocked)

## Rule 5: Domain Allowlist

Only domains explicitly listed in `allowed_domains` can be called. Any tool calling an unlisted domain is blocked at both creation and execution time.

In [ ]:
print('RULE 5: Domain Allowlist')
with tempfile.TemporaryDirectory() as d:
    pc = PolicyConfig(allowed_domains=['api.github.com', 'jsonplaceholder.typicode.com'])
    m = await Matimo.init([d], auto_discover=True, policy_config=pc)
    test_urls = [
        ('api.github.com (allowed)', 'https://api.github.com/zen', True),
        ('evil.attacker.com (blocked)', 'https://evil.attacker.com/steal', False),
        ('jsonplaceholder.typicode.com (allowed)', 'https://jsonplaceholder.typicode.com/todos/1', True),
    ]
    for label, url, should_pass in test_urls:
        try:
            await m.execute('matimo_web_fetch', {'url': url, 'method': 'GET'})
            outcome = 'ALLOWED'
        except Exception:
            outcome = 'BLOCKED'
        expected = 'ALLOWED' if should_pass else 'BLOCKED'
        status = '' if outcome == expected else ' (UNEXPECTED!)'
        print(f'  {outcome}{status} — {label}')

## Rule 6: HTTP Method Control

By default only GET and POST are permitted. Tools using PUT, DELETE, PATCH require explicit `allowed_http_methods` config.

In [ ]:
print('RULE 6: HTTP Method Control')
with tempfile.TemporaryDirectory() as d:
    pc_strict = PolicyConfig(allowed_http_methods=['GET'], allowed_domains=['api.example.com'])
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d], policy_config=pc_strict)
    for method in ['GET', 'POST', 'DELETE', 'PUT']:
        yaml = f"""name: method_{method.lower()}
description: test {method}
version: '1.0.0'
execution:
  type: http
  url: https://api.example.com/resource
  method: {method}
parameters:
  type: object
  properties: {{}}
"""
        with open(os.path.join(d, f'method_{method.lower()}.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        loaded = any(t.name == f'method_{method.lower()}' for t in m.list_tools())
        expected_allowed = method in ['GET']
        status = 'ALLOWED' if loaded else 'BLOCKED'
        expected = 'ALLOWED' if expected_allowed else 'BLOCKED'
        flag = '' if status == expected else ' (UNEXPECTED!)'
        print(f'  {status}{flag} — {method}')

## Rule 7: Credential Allowlist

Tools that reference auth environment variables (API keys, tokens) must declare them in `allowed_credentials`. Undeclared credentials cause the tool to be blocked or queued for human approval.

In [ ]:
print('RULE 7: Credential Allowlist')
with tempfile.TemporaryDirectory() as d:
    pc = PolicyConfig(
        allowed_credentials=['GITHUB_TOKEN'],
        allowed_domains=['api.github.com']
    )
    m = await Matimo.init([d], auto_discover=True, untrusted_paths=[d], policy_config=pc)
    cred_tests = [
        ('GITHUB_TOKEN (allowlisted)', 'GITHUB_TOKEN', True),
        ('SECRET_KEY (not allowlisted)', 'SECRET_KEY', False),
        ('AWS_SECRET_ACCESS_KEY (not allowlisted)', 'AWS_SECRET_ACCESS_KEY', False),
    ]
    for label, cred_var, should_pass in cred_tests:
        yaml = f"""name: cred_test
description: credential test
version: '1.0.0'
execution:
  type: http
  url: https://api.github.com/user
  method: GET
  headers:
    Authorization: Bearer ${{{cred_var}}}
parameters:
  type: object
  properties: {{}}
"""
        with open(os.path.join(d, 'cred_test.yaml'), 'w') as f: f.write(yaml)
        await m.reload()
        loaded = any(t.name == 'cred_test' for t in m.list_tools())
        status = 'ALLOWED' if loaded else 'BLOCKED'
        expected = 'ALLOWED' if should_pass else 'BLOCKED'
        flag = '' if status == expected else ' (UNEXPECTED!)'
        print(f'  {status}{flag} — {label}')

## Rules 8 & 9: Production Risk Gate and Draft Tool Gate

In `prod` environment: tools with `high` or `critical` risk are blocked at execution time. Draft (unapproved) tools require admin role or are rejected outright.

In [ ]:
print('RULES 8 & 9: Production Environment Gates')
print()
print('RULE 8: High-risk tool in prod environment')
with tempfile.TemporaryDirectory() as d:
    m_prod = await Matimo.init(
        [d], auto_discover=True, untrusted_paths=[d],
        policy_config=PolicyConfig(
            allowed_domains=['dangerous.example.com'],
            allowed_http_methods=['DELETE'],
        )
    )
    high_risk_yaml = """name: delete_all
description: delete everything
version: '1.0.0'
execution:
  type: http
  url: https://dangerous.example.com/delete
  method: DELETE
parameters:
  type: object
  properties: {}
"""
    with open(os.path.join(d, 'delete_all.yaml'), 'w') as f: f.write(high_risk_yaml)
    await m_prod.reload()
    try:
        await m_prod.execute('delete_all', {}, context={'environment': 'prod'})
        print('  ALLOWED (unexpected!)')
    except Exception as e:
        print('  BLOCKED by production risk gate')

print()
print('RULE 9: Draft tool gate (unapproved tool in prod)')
with tempfile.TemporaryDirectory() as d:
    m2 = await Matimo.init([d], auto_discover=True, untrusted_paths=[d])
    draft_yaml = """name: draft_tool
description: not yet approved
version: '1.0.0'
status: draft
execution:
  type: http
  url: https://jsonplaceholder.typicode.com/todos/1
  method: GET
parameters:
  type: object
  properties: {}
"""
    with open(os.path.join(d, 'draft_tool.yaml'), 'w') as f: f.write(draft_yaml)
    await m2.reload()
    try:
        await m2.execute('draft_tool', {}, context={'environment': 'prod', 'roles': []})
        print('  ALLOWED (unexpected!)')
    except Exception as e:
        print('  BLOCKED by draft tool gate')

---
## Summary

You just ran all 9 deterministic security rules against live code. No LLM. No API key. Pure policy enforcement.

| Rule | Status |
|------|--------|
| 1. SSRF Protection | Blocks 169.254.x, 127.x, 10.x, 192.168.x, 172.16-31.x |
| 2. Protected Namespace | Blocks `matimo_*` name collisions |
| 3. Command Execution Block | Blocks `execution.type: command` |
| 4. Function Execution Block | Blocks `execution.type: function` |
| 5. Domain Allowlist | Blocks calls to non-allowlisted domains |
| 6. HTTP Method Control | Blocks PUT/DELETE/PATCH unless explicitly allowed |
| 7. Credential Allowlist | Blocks undeclared auth env vars |
| 8. Production Risk Gate | Blocks high/critical tools in prod |
| 9. Draft Tool Gate | Blocks unapproved tools in prod without admin role |

**Next:** `03_meta_tools.ipynb` — Watch an agent create its own tools at runtime.

GitHub: https://github.com/tallclub/matimo